# 48 — ChromaDB
**Goal:** Use ChromaDB for persistent vector storage with metadata filtering.

Chapter 47 ended with a fast vector index and an open gap: FAISS stores vectors, but not the structured facts attached to them — role, years of experience, skills. ChromaDB is an embedded, open-source vector database that keeps embeddings *and* their metadata together, persists them to disk, and filters queries by both at once.

**Why it matters for resumes / ATS:** real talent search is never "similar to this text" alone; it is "similar *and* senior *and* in Berlin". ChromaDB's `where` filters make that a single query rather than a two-stage post-filter, and its persistence means the resume corpus survives restarts without a re-embedding job. It is the pragmatic middle ground between raw FAISS and a hosted vector DB like Pinecone or Weaviate.

## 1. Setting Up ChromaDB

ChromaDB is a client-style library that also runs fully **embedded** — a plain Python process, no external service to install. The unit of organization is a **collection**, roughly a named table of vectors.

**What the code does:**
- Creates a `chromadb.Client` with `Settings(anonymized_telemetry=False)` to keep usage telemetry off.
- Tries `create_collection("resumes")` and falls back to `get_collection("resumes")` on error, so re-running the cell is harmless (idempotent setup).

**Try it:** the try/except pattern is the standard way to handle "collection may already exist" in scripts and notebooks. In production you would typically version collection names (`resumes_v2`) or recreate deliberately, rather than silently reusing stale data.

In [ ]:
import chromadb
from chromadb.config import Settings

# In-memory for development
client = chromadb.Client(Settings(anonymized_telemetry=False))
try:
    collection = client.create_collection("resumes")
    print(f"Collection 'resumes' created")
except:
    collection = client.get_collection("resumes")
    print("Collection already exists")

## 2. Adding Resumes with Metadata

The `add()` call is where ChromaDB earns its keep: one call ingests documents, their embeddings (computed automatically by ChromaDB's default embedding function when you pass text), and arbitrary metadata as a dict per row.

**What the code does:** adds three resumes with:
- `documents` — the raw resume text,
- `metadatas` — structured dicts: `role`, `years`, and a comma-joined `skills` string,
- `ids` — stable string keys (`resume_001` ...) that later `get` / `update` / `delete` calls use to address rows.

**Expected:** `collection.count()` returns 3. Notice the metadata schema is arbitrary — you could store `location`, `salary_expectation`, `last_updated`, anything you want to query later. That flexibility is what FAISS from Ch. 47 lacks out of the box.

In [ ]:
# Add resume embeddings with metadata
collection.add(
    documents=[
        "Senior data scientist with Python, NLP, TensorFlow. 5 years experience.",
        "Java backend engineer with Spring Boot, microservices, 3 years.",
        "Frontend developer with React, TypeScript, 2 years experience.",
    ],
    metadatas=[
        {"role": "data_scientist", "years": 5, "skills": "python,nlp,tensorflow"},
        {"role": "backend", "years": 3, "skills": "java,spring,microservices"},
        {"role": "frontend", "years": 2, "skills": "react,typescript"},
    ],
    ids=["resume_001", "resume_002", "resume_003"],
)
print(f"Collection has {collection.count()} documents")

## 3. Querying with Metadata Filtering

The payoff for storing metadata: `collection.query()` accepts `query_texts` (which it embeds internally) and a `where` filter applied **during** the search, not after it. Only vectors matching the filter are candidates, so the result count can be smaller than `n_results`.

**What the code does:**
- Queries with `query_texts=["looking for NLP expert with Python"]`, `n_results=2`, and `where={"role": "data_scientist"}`.
- Prints each returned document, its metadata, and its distance.

**Expected behavior:** exactly one document matches the filter — `resume_001` is the only row with `role == "data_scientist"` — so the loop prints a single result even though `n_results=2`; its distance reflects how close that resume text is to the query. Filters work on any metadata key, with numeric comparisons (e.g. `years > 3`) supported alongside string equality.

In [ ]:
# Query: find data scientists with similarity
results = collection.query(
    query_texts=["looking for NLP expert with Python"],
    n_results=2,
    where={"role": "data_scientist"},
)
print("Filtered query results:")
for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0], results['metadatas'][0], results['distances'][0]
)):
    print(f"  {i+1}. [{metadata['role']}] {doc[:50]:50s} (dist: {distance:.3f})")

## 4. Updating and Deleting

Resumes change — candidates gain skills, move jobs, or withdraw. ChromaDB supports `update()` to replace a row's document and metadata, `get()` to fetch rows by id, and `delete()` to remove them.

**What the code does:**
- `update()` rewrites `resume_001` with a new document and metadata (`years` bumped from 5 to 6, `skills` trimmed) — the count is unchanged because the id is preserved.
- `get(ids=["resume_001"], include=["documents", "metadatas"])` fetches that row back to verify the new content.

**Expected:** the count stays 3 after the update, and the `get` returns the six-year version. The heading promises deletion too — in ChromaDB that is `collection.delete(ids=[...])`; it is not exercised here but follows the same id-addressed pattern.

In [ ]:
# Update a resume
collection.update(
    documents=["Senior data scientist with 6 years experience, Python, NLP."],
    ids=["resume_001"],
    metadatas=[{"role": "data_scientist", "years": 6, "skills": "python,nlp"}],
)
print(f"Updated resume_001. Collection count: {collection.count()}")

# Get by ID
result = collection.get(ids=["resume_001"], include=["documents", "metadatas"])
print(f"Get resume_001: {result['documents'][0][:40]}... [{result['metadatas'][0]}]")

## Summary: ChromaDB adds metadata filtering on top of vector search. Good for production MVPs.

ChromaDB delivers what FAISS left open in Ch. 47: embeddings, metadata, and persistence in one embedded package. `create_collection` / `add` / `query` / `update` / `get` cover the full CRUD cycle, and `where` filters make hybrid searches ("similar text AND role = data scientist") a single call. Because it runs in-process with zero infrastructure, it is the fastest path from prototype to a real ATS backend — you can swap in a hosted vector DB later without changing the shape of your queries.

## Key Insight

**Vectors are only half the database — metadata is what makes search usable.**

A resume corpus queried purely by similarity returns noise; filtering by role, years, or skills is what turns retrieval into recruiting. ChromaDB's contribution is combining both in one query and persisting the result, at the cost of nothing but a pip install. That metadata awareness previews how a real ATS organizes candidates, and it sets up the question Ch. 49 answers: how do we know the similarity scores underneath are any good?